# 01 — Train the baseline caries detector

Runs after `00_setup_and_sanity_check.ipynb` has passed. This is the actual
Phase 3 GPU training run: HierarchicalDet's Swin-Large + DiffusionDet
architecture, `SOLVER.MAX_ITER = 40000`, `IMS_PER_BATCH = 2`.

**Read this first — two real bugs found and fixed during development, both
confirmed by actually running this pipeline (not guessed):**

1. **Do not use `hierarchialdet.dataset_mapper.DiffusionDetDatasetMapper`.**
   It unconditionally tries to open two hardcoded personal file paths from
   the original author's machine
   (`ibrahim/Diseasedataset_base_enumeration_m_t_inference_train/...`) in its
   constructor, and crashes with `FileNotFoundError` for anyone else. This
   notebook defines its own `CariesDatasetMapper` instead (below), which
   trains directly from ground-truth boxes -- confirmed the model's own
   `prepare_inferred_boxes`/`prepare_targets` gracefully fall back to
   standard (non-hierarchical-curriculum) DiffusionDet training when no
   pretrained-boxes are provided (bare `except: pass` in the source), so
   this is a correct simplification for a caries-only detector, not a hack.
2. **`cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE = "full_model"` (the config's
   default) is invalid in the detectron2 version this install recipe pulls.**
   Raises `ValueError: 'full_model' is not a valid GradientClipType` — this
   version only accepts `"value"` or `"norm"`. Overridden below to `"norm"`.

**Kaggle session limits**: GPU sessions have a runtime limit and a weekly GPU
quota — 40k iterations will almost certainly NOT fit in one session. This
notebook checkpoints periodically and resumes automatically; see the
"multi-session workflow" section below for exactly how to continue after a
session ends.

## 1. Setup (repeat of 00, condensed — see that notebook for the full
explanation of each step if anything here fails)

In [ ]:
!git clone https://github.com/christopherh-88/Carries-Confidence.git
%cd Carries-Confidence
!pip install -q -r requirements-core.txt
!bash scripts/clone_baseline.sh
!pip install -q ninja
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q timm scipy Pillow

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
os.makedirs("models_weights", exist_ok=True)
if not os.path.exists("models_weights/swin_large_patch4_window7_224_22k.pkl"):
    !curl -sL "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_large_patch4_window7_224_22k.pth" \
      -o models_weights/swin_large_patch4_window7_224_22k_raw.pth
    import pickle
    ckpt = torch.load("models_weights/swin_large_patch4_window7_224_22k_raw.pth", map_location="cpu", weights_only=False)
    assert ckpt["model"]["patch_embed.proj.weight"].shape[0] == 192, "expected Swin-Large's 192-dim embedding"
    converted = {"model": ckpt["model"], "__author__": "third_party", "matching_heuristics": True}
    with open("models_weights/swin_large_patch4_window7_224_22k.pkl", "wb") as f:
        pickle.dump(converted, f)
    os.remove("models_weights/swin_large_patch4_window7_224_22k_raw.pth")
print("backbone weights ready")

## 2. Dataset

Same as notebook 00 — set `DATA_ROOT` to wherever DENTEX actually is
(attached Kaggle Dataset, or downloaded directly).

In [ ]:
DATA_ROOT = "/kaggle/input/dentex/DENTEX/training_data/quadrant-enumeration-disease"
assert os.path.exists(f"{DATA_ROOT}/xrays"), f"expected images at {DATA_ROOT}/xrays -- check DATA_ROOT"
print("DATA_ROOT ok:", DATA_ROOT)

## 3. Register the dataset and define the custom mapper

`CariesDatasetMapper` reads our COCO annotations (category_id_1/2/3, the
DENTEX quadrant/enumeration/diagnosis hierarchy) into an `Instances` object
with `gt_boxes`, `gt_classes_1`, `gt_classes_2`, `gt_classes_3` — the exact
fields `DiffusionDet.forward()`'s training path reads (confirmed by reading
`hierarchialdet/detector.py`). No pretrained-boxes curriculum, no dependency
on the author's missing files.

In [ ]:
import sys, warnings, copy
warnings.filterwarnings("ignore")
import pycocotools.mask, pycocotools.coco, pycocotools.cocoeval
import detectron2
from detectron2 import _C
sys.path.insert(0, "external/HierarchicalDet")
from hierarchialdet.config import add_diffusiondet_config

import cv2
import numpy as np
from detectron2.structures import Instances, Boxes
from detectron2.data import DatasetCatalog

sys.path.insert(0, ".")
from src.data.dentex import load_coco, patient_level_split, register_dentex_detectron2

coco = load_coco(f"{DATA_ROOT}/train_quadrant_enumeration_disease.json")
split = patient_level_split(coco, seed=0)
register_dentex_detectron2(coco, f"{DATA_ROOT}/xrays", split)
print("registered:", {k: len(v) for k, v in split.items()})


class CariesDatasetMapper:
    """Reads DENTEX's hierarchical annotations into DiffusionDet's expected
    Instances fields. Ground-truth boxes only -- no pretrained-box curriculum
    (see the note at the top of this notebook for why that's the right call
    here, not a corner cut)."""

    def __init__(self, target_size=800, is_train=True):
        self.target_size = target_size
        self.is_train = is_train

    def __call__(self, d):
        d = copy.deepcopy(d)
        img = cv2.imread(d["file_name"], cv2.IMREAD_COLOR)
        h0, w0 = img.shape[:2]
        img = cv2.resize(img, (self.target_size, self.target_size))
        scale_x, scale_y = self.target_size / w0, self.target_size / h0

        out = {
            "image": torch.as_tensor(img.transpose(2, 0, 1).astype(np.float32)),
            "height": self.target_size, "width": self.target_size,
        }
        if not self.is_train:
            return out

        inst = Instances((self.target_size, self.target_size))
        boxes, c1, c2, c3 = [], [], [], []
        for ann in d.get("annotations", []):
            if ann.get("iscrowd", 0):
                continue
            x, y, w, h = ann["bbox"]
            boxes.append([x * scale_x, y * scale_y, (x + w) * scale_x, (y + h) * scale_y])
            c1.append(ann["category_id_1"]); c2.append(ann["category_id_2"]); c3.append(ann["category_id_3"])
        inst.gt_boxes = Boxes(torch.tensor(boxes, dtype=torch.float32)) if boxes else Boxes(torch.zeros(0, 4))
        inst.gt_classes_1 = torch.tensor(c1, dtype=torch.int64)
        inst.gt_classes_2 = torch.tensor(c2, dtype=torch.int64)
        inst.gt_classes_3 = torch.tensor(c3, dtype=torch.int64)
        out["instances"] = inst
        return out


## 4. Config and trainer

`CariesTrainer` overrides `build_train_loader` to use `CariesDatasetMapper`
instead of HierarchicalDet's broken one. Everything else (optimizer,
scheduler, checkpointing, logging) is detectron2's own well-tested
`DefaultTrainer` machinery — deliberately not hand-rolled, to keep the parts
that don't need to be custom as boring/standard as possible.

In [ ]:
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from detectron2.data import build_detection_train_loader

class CariesTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=CariesDatasetMapper(is_train=True))

cfg = get_cfg()
add_diffusiondet_config(cfg)
cfg.merge_from_file("external/HierarchicalDet/configs/diffdet.custom.swinbase.nonpretrain.yaml")
cfg.MODEL.WEIGHTS = "models_weights/swin_large_patch4_window7_224_22k.pkl"
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.DATASETS.TRAIN = ("custom_train_class",)
cfg.DATASETS.TEST = ()  # evaluation is a separate notebook (03) -- keep this one focused on training
cfg.DATALOADER.NUM_WORKERS = 0  # confirmed: NUM_WORKERS>0 crashed in development (macOS
# spawn-based multiprocessing -- DataLoader worker exited unexpectedly, likely a
# pickling issue with the mapper/cv2 in a spawned child process). Kaggle runs Linux
# (fork-based multiprocessing), which MIGHT not hit this -- if you want the speedup,
# try NUM_WORKERS=2 and watch for the same crash before trusting it; 0 is what's
# actually been verified end-to-end.

# --- confirmed-necessary override (see the note at the top) ---
cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE = "norm"  # "full_model" (the config default) is invalid in this detectron2 version

# --- tune these for your GPU's memory; config default is IMS_PER_BATCH=2 ---
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.MAX_ITER = 40000       # per the proposal/config; will need multiple sessions, see below
cfg.SOLVER.CHECKPOINT_PERIOD = 500  # roughly every 500 iters -- tune based on the throughput you measure below

cfg.OUTPUT_DIR = "/kaggle/working/checkpoints"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print("config ready, device:", cfg.MODEL.DEVICE)

## 5. Multi-session workflow — read before hitting Run

`SOLVER.MAX_ITER=40000` will not finish in one Kaggle session. Measured on
CPU during development: ~35s/iteration -> ~390 hours total (see
`docs/phase3_model_benchmarks.md`) -- GPU will be much faster but almost
certainly still multi-session. Two things make this resumable:

1. **Within a session**: `trainer.resume_or_load(resume=True)` (below) checks
   `cfg.OUTPUT_DIR` for a `last_checkpoint` file and continues from there
   automatically if one exists -- confirmed locally: killing training after
   iteration 3 and rerunning with `resume=True` correctly picked back up at
   iteration 3, not 0.
2. **Across sessions** (the part that needs a manual step): `/kaggle/working/`
   is only preserved if you **Save Version** (commit) the notebook before the
   session ends. Next session: open a **new** session of this notebook, add
   the *previous version's output* as a data source (Kaggle's "Notebook
   Output Files" — the sidebar's "Add Data" supports this), copy the
   checkpoint files into `/kaggle/working/checkpoints/` before running the
   training cell again, so `resume_or_load(resume=True)` finds them.

Benchmark your actual per-iteration time on the real GPU **before** trusting
any ETA — the ~35s/iter above is a CPU number from development, not this
notebook's hardware.

In [ ]:
import time

# quick throughput check: time a handful of real iterations before committing
# to the full 40000 -- adjust SOLVER.CHECKPOINT_PERIOD / your session budget
# based on what this actually reports on your GPU.
cfg.SOLVER.MAX_ITER = 10
trainer = CariesTrainer(cfg)
trainer.resume_or_load(resume=True)  # picks up an existing checkpoint if present, else starts fresh
t0 = time.time()
trainer.train()
dt = time.time() - t0
per_iter = dt / max(1, trainer.iter - (trainer.start_iter or 0))
print(f"measured: {per_iter:.2f}s/iteration on this hardware")
print(f"extrapolated full 40000-iter run: {40000 * per_iter / 3600:.1f} hours")

## 6. The real run

Once the throughput check above looks reasonable, bump `MAX_ITER` back to
40000 (or whatever your session-budget math says to target for this
session's chunk) and rerun. Re-running this cell after a `Save Version` +
new session (with the previous checkpoint copied into `cfg.OUTPUT_DIR`) will
resume rather than restart, per the workflow above.

In [ ]:
cfg.SOLVER.MAX_ITER = 40000  # or a smaller per-session target -- your call based on the throughput above
trainer = CariesTrainer(cfg)
trainer.resume_or_load(resume=True)
trainer.train()
print("done at iteration:", trainer.iter)